# Exercise: Classify Clothing Images with a CNN

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) has the same format as MNIST: 70,000 greyscale images of 28 × 28 pixels in ten classes. The images show clothing and shoes from the Zalando online shop instead of digits, and they are harder to classify: a shirt, a T-shirt and a pullover look much alike at this resolution.

In this exercise you build a dense baseline and a convolutional network, diagnose their learning curves, reduce overfitting with dropout and early stopping, and evaluate your chosen model once on the test set. The tasks describe what to build and what to report; the code is yours to write. Notebooks 01 and 02 contain every layer and function you need. A CNN trains for about two minutes, so plan your runs before you start them.

## Learning Objectives

By the end of this exercise, you should be able to:

- Build a dense baseline and a CNN for Fashion-MNIST and compare them on a validation set.
- Calculate the output shapes and parameters of a CNN and check them against `model.summary()`.
- Diagnose overfitting in the learning curves of a CNN and reduce it with dropout and early stopping.
- Evaluate the chosen model on the test set and identify the classes it confuses most.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## Step 1: Load the Data

[`keras.datasets.fashion_mnist.load_data()`](https://keras.io/api/datasets/fashion_mnist/) returns 60,000 training and 10,000 test images and their labels from 0 to 9; it downloads them on the first run. The cell below loads the data and shows one example of each class.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
print("Training images:", X_train_full.shape, " test images:", X_test.shape)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for label, ax in enumerate(axes):
    ax.imshow(X_train_full[y_train_full == label][0], cmap="gray")
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
plt.show()

## Step 2: Prepare the Data

- Split the training images into a training set and a validation set of 10,000 images, with `stratify` and `random_state=42`.
- Add a channel axis to the training, validation and test images, so each set has the shape `(images, 28, 28, 1)`.
- Print the shapes of the three sets.

Scale the pixels inside every model with a `Rescaling(1 / 255)` layer, as in notebook 02.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=10_000, stratify=y_train_full, random_state=42
)
X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test = X_test[..., np.newaxis]
print("Training:", X_train.shape, " validation:", X_valid.shape, " test:", X_test.shape)

## Step 3: Plan How You Train

You will train four or five models. How you organise the code is up to you:

- **Write helper functions** (optional), similar to those in notebook 02. For example, `compile_and_fit(model, epochs, callbacks=None)` compiles and trains a model and returns the history as a DataFrame, and `plot_curves(logs, title)` draws the loss next to the accuracy.
- **Write each model out in full**, and copy and change the cells from one step to the next.

Whichever way you choose, keep these settings the same for every model, so the comparison is fair and your numbers are close to the solution's:

- Call `keras.utils.clear_session()` and `keras.utils.set_random_seed(42)` before you build each model.
- Compile with Adam (learning rate 0.001), the `sparse_categorical_crossentropy` loss and the accuracy metric.
- Train with `batch_size=128` and the validation set as `validation_data`.
- Plot the loss and the accuracy curves of every run, for both the training and the validation set.

Also create an empty dictionary `results`. After every model, store its validation accuracy there under a descriptive name, so you can compare all models in step 8.

The cell below is for your helper functions, if you write them; you can also leave it empty.

In [ ]:
def compile_and_fit(model, epochs, callbacks=None):
    """Compile a classifier with Adam and train it; return the learning curves as a DataFrame."""
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    history = model.fit(
        X_train, y_train, epochs=epochs, batch_size=128, validation_data=(X_valid, y_valid),
        callbacks=callbacks, verbose=0,
    )
    return pd.DataFrame(history.history)


def plot_curves(logs, title):
    """Plot the training and validation loss and accuracy of one run."""
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    logs[["loss", "val_loss"]].plot(ax=ax_loss, xlabel="Epoch", title=f"{title}: loss")
    logs[["accuracy", "val_accuracy"]].plot(ax=ax_acc, xlabel="Epoch", title=f"{title}: accuracy")
    plt.show()


results = {}

## Step 4: A Dense Baseline

Build a dense network: the input, a `Rescaling` layer, `Flatten`, one hidden layer of 128 ReLU neurons and a softmax output layer with one neuron per class. Train it for 20 epochs, plot the learning curves and store the validation accuracy.

How many parameters does it have? Does it overfit?

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
dense_model = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Rescaling(1 / 255),
        keras.layers.Flatten(),
        keras.layers.Dense(units=128, activation="relu"),
        keras.layers.Dense(units=10, activation="softmax"),
    ]
)
dense_logs = compile_and_fit(dense_model, epochs=20)
plot_curves(dense_logs, "Dense network")
results["Dense network"] = dense_model.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Parameters: {dense_model.count_params():,}, validation accuracy: {results['Dense network']:.4f}")

**Result.** The dense network has 101,770 parameters and reaches a validation accuracy of 0.896. It overfits after about 10 epochs: the validation loss stops falling at about 0.31 while the training loss keeps falling, and the training accuracy rises to 0.93 while the validation accuracy levels off near 0.89 to 0.90.

## Step 5: A Convolutional Network

Build a CNN with two convolutional blocks:

1. `Conv2D` with 32 filters of 3 × 3 and ReLU, followed by `MaxPooling2D` with `pool_size=2`,
2. `Conv2D` with 64 filters of 3 × 3 and ReLU, followed by `MaxPooling2D` with `pool_size=2`,

then `Flatten`, a dense layer of 128 ReLU neurons and the softmax output layer. Use the default `padding="valid"`.

**Before you train it**, calculate the output shape and the number of parameters of every layer with the formulas of notebook 01, and write them in the Markdown cell below. Remember that pooling rounds down: an odd size such as 11 becomes 5. Then print `model.summary()` and compare. Which layer has the most parameters?

Train the CNN for 15 epochs, plot the learning curves and store the validation accuracy. At which epoch is the validation loss lowest, and what happens afterwards?

In [ ]:
def build_cnn(dropout_rate=0.0):
    """Return the two-block CNN, with optional dropout before and after the dense hidden layer."""
    keras.utils.clear_session()
    keras.utils.set_random_seed(42)
    layers = [
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Rescaling(1 / 255),
        keras.layers.Conv2D(filters=32, kernel_size=3, activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Conv2D(filters=64, kernel_size=3, activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
    ]
    if dropout_rate > 0:
        layers.append(keras.layers.Dropout(dropout_rate))
    layers.append(keras.layers.Dense(units=128, activation="relu"))
    if dropout_rate > 0:
        layers.append(keras.layers.Dropout(dropout_rate))
    layers.append(keras.layers.Dense(units=10, activation="softmax"))
    return keras.Sequential(layers)


cnn = build_cnn()
cnn.summary()

**Calculation.**

| Layer | Output shape | Parameters |
|---|---|---|
| `Conv2D`, 32 filters 3 × 3 | 26 × 26 × 32 | $(3 \cdot 3 \cdot 1 + 1) \cdot 32 = 320$ |
| `MaxPooling2D` | 13 × 13 × 32 | 0 |
| `Conv2D`, 64 filters 3 × 3 | 11 × 11 × 64 | $(3 \cdot 3 \cdot 32 + 1) \cdot 64 = 18{,}496$ |
| `MaxPooling2D` | 5 × 5 × 64 (11 / 2 rounded down) | 0 |
| `Flatten` | 1,600 | 0 |
| `Dense`, 128 | 128 | $1{,}600 \cdot 128 + 128 = 204{,}928$ |
| `Dense`, 10 | 10 | $128 \cdot 10 + 10 = 1{,}290$ |

The total of 225,034 matches `model.summary()`. The first dense layer holds 91 % of the parameters; the two convolutional layers together have fewer than 19,000.

In [ ]:
cnn_logs = compile_and_fit(cnn, epochs=15)
plot_curves(cnn_logs, "CNN")
results["CNN"] = cnn.evaluate(X_valid, y_valid, verbose=0)[1]
best_epoch = cnn_logs["val_loss"].idxmin() + 1  # epochs are counted from 1
print(f"Validation accuracy: {results['CNN']:.4f}; lowest validation loss "
      f"{cnn_logs['val_loss'].min():.3f} at epoch {best_epoch}, {cnn_logs['val_loss'].iloc[-1]:.3f} at epoch 15")

**Result.** The CNN reaches a validation accuracy of 0.911, better than the dense network's 0.896. Its validation loss is lowest at epoch 8 (0.245) and rises to 0.289 by epoch 15, while the training loss keeps falling and the training accuracy climbs to 0.96. The validation accuracy stays flat at about 0.91 from epoch 6 onwards. This is panel C of the learning-curves figure: after epoch 8 the network mainly memorises the training set.

## Step 6: Add Dropout

Build the same CNN with a `Dropout` layer with a rate of 0.3 after `Flatten` and another after the dense hidden layer. Train it for 15 epochs, plot the learning curves and store the validation accuracy.

Compare the curves with those of step 5: has the gap between training and validation become smaller?

In [ ]:
dropout_cnn = build_cnn(dropout_rate=0.3)
dropout_logs = compile_and_fit(dropout_cnn, epochs=15)
plot_curves(dropout_logs, "CNN with dropout")
results["CNN with dropout"] = dropout_cnn.evaluate(X_valid, y_valid, verbose=0)[1]
pd.Series(results).round(4)

**Result.** With dropout the validation accuracy rises to 0.922, the best so far. The curves look very different from those of step 5: the validation loss keeps falling to about 0.22 over all 15 epochs, and the training and validation accuracy stay close together (0.93 and 0.92 at epoch 15). The validation curves lie above the training curves for most epochs, because the training metrics are computed with dropout switched on. The accuracy is still rising at epoch 15, so the network can benefit from more epochs.

## Step 7: Train Longer and Keep the Best Epoch

Train the CNN with dropout again, this time for up to 30 epochs, with two callbacks:

- [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) that monitors the validation accuracy with `patience=5` and `restore_best_weights=True`,
- [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/) that saves the best model to `models/fashion_cnn_best.keras`, monitoring the same value.

Create the `models` folder first. Load the saved model with [`keras.saving.load_model()`](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/#loadmodel-function), store its validation accuracy and plot the learning curves. After how many epochs did training stop?

In [ ]:
Path("models").mkdir(exist_ok=True)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("models/fashion_cnn_best.keras", monitor="val_accuracy", save_best_only=True),
]
final_cnn = build_cnn(dropout_rate=0.3)
final_logs = compile_and_fit(final_cnn, epochs=30, callbacks=callbacks)
print(f"Trained {len(final_logs)} epochs; best validation accuracy at epoch {final_logs['val_accuracy'].idxmax() + 1}")
plot_curves(final_logs, "CNN with dropout and early stopping")

best_cnn = keras.saving.load_model("models/fashion_cnn_best.keras")
results["CNN with dropout, early stopping"] = best_cnn.evaluate(X_valid, y_valid, verbose=0)[1]

**Result.** With early stopping, training stopped after 20 epochs: the validation accuracy was highest at epoch 15 and did not improve in the next five. The model restored from the checkpoint is therefore the same as the 15-epoch model of step 6, with the same validation accuracy of 0.922. The extra epochs did not help here, but the callbacks found this out without you having to guess the number of epochs, and the best model is saved on disk.

## Step 8: Compare the Models

Show the validation accuracies of all your models in one table, sorted from best to worst, and choose your final model.

In [ ]:
pd.Series(results, name="Validation accuracy").sort_values(ascending=False).round(4).to_frame()

**Result.** The CNN with dropout is the best model (0.922), followed by the CNN without dropout (0.911) and the dense network (0.896). The early-stopping run gives the same model as step 6, so either can be chosen; the saved checkpoint is the one to keep, because it can be loaded later without training again.

## Step 9: Evaluate Once on the Test Set

Evaluate your chosen model on the test set. The choice is already made, so also report the dense baseline's test accuracy for reference only. Then plot the confusion matrix of your chosen model on the test set with the class names as labels, and show a few misclassified test images with their true and predicted class.

Which classes does your model confuse most? Looking at the example images in step 1, why could that be?

In [ ]:
test_accuracy = {
    "Dense network": dense_model.evaluate(X_test, y_test, verbose=0)[1],
    "CNN with dropout, early stopping": best_cnn.evaluate(X_test, y_test, verbose=0)[1],
}
print(pd.Series(test_accuracy, name="Test accuracy").round(4).to_frame())

y_pred = best_cnn.predict(X_test, verbose=0).argmax(axis=1)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names, xticks_rotation=45, colorbar=False, ax=ax
)
ax.grid(False)
plt.show()

wrong = np.flatnonzero(y_pred != y_test)
fig, axes = plt.subplots(1, 8, figsize=(15, 2.4))
for ax, i in zip(axes, wrong):
    ax.imshow(X_test[i, :, :, 0], cmap="gray")
    ax.set_title(f"{class_names[y_test[i]]}\npredicted: {class_names[y_pred[i]]}", fontsize=8)
    ax.axis("off")
plt.show()

**Result.** The chosen CNN reaches a test accuracy of 0.916, close to its validation accuracy of 0.922. For reference, the dense network reaches 0.878. The CNN misclassifies 8.4 % of the test images, the dense network 12.2 %.

The confusion matrix shows where the remaining errors are. Trousers, bags, sandals, sneakers and ankle boots are almost always right (above 0.96). The weakest class by far is shirt: only 693 of the 1,000 test shirts are classified correctly, and 151 are predicted as T-shirt/top, 61 as coat and 49 as pullover. The confusions also go the other way: 67 T-shirts and 62 pullovers are predicted as shirts. The example images in step 1 explain why: at 28 × 28 pixels in greyscale, a shirt, a T-shirt, a pullover and a coat all have the same outline of a body with sleeves, and the details that tell them apart, such as a collar or buttons, cover only a few pixels.

## Reflection

Answer these questions in the Markdown cell below.

1. Compare the parameter counts of the dense network and the CNN, layer by layer. Where are most of the CNN's parameters, and why?
2. Did the CNN without dropout overfit? Which signs in the learning curves show it, and what did dropout change?
3. Your final model still confuses some classes. Would a larger CNN solve this, or is part of the problem in the data itself?
4. Every model was compared on the validation set, and the test set only reported the accuracy of the chosen model. What would go wrong if you had chosen the model by its test accuracy?

**Answers.**

1. The dense network has 101,770 parameters, almost all in its hidden layer (784 inputs × 128 neurons). The CNN has 225,034, and 204,928 of them, 91 %, are in its first dense layer, which connects the 1,600 values of the last feature maps to 128 neurons. The two convolutional layers have only 18,816 parameters, because each filter is small and is reused at every position of the image.
2. Yes. Its validation loss was lowest at epoch 8 and then rose, while the training loss kept falling and the gap between training and validation accuracy grew to about 0.05. With dropout the validation loss kept falling, the two accuracy curves stayed close together, and the validation accuracy rose from 0.911 to 0.922.
3. Only partly. A larger or longer-trained CNN might reduce the confusion slightly, but most of it is in the data: at 28 × 28 pixels in greyscale, shirts, T-shirts, pullovers and coats share the same outline, and some images would be ambiguous for a person as well. Higher-resolution or colour images would help more than a larger network.
4. The test accuracy would then be part of the choice, so it would be optimistic in the same way the validation accuracy is: you would pick the model that happens to suit these 10,000 test images. You would have no unbiased estimate left of how the model performs on new images.

## References

- Xiao, H., Rasul, K. and Vollgraf, R. (2017). [Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms](https://arxiv.org/abs/1708.07747).
- [Fashion-MNIST on GitHub](https://github.com/zalandoresearch/fashion-mnist): the dataset, its classes and a benchmark of many models.
- TensorFlow tutorial: [Basic classification: Classify images of clothing](https://www.tensorflow.org/tutorials/keras/classification), a dense network on the same data.
- Keras API: [`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/), [`Dropout`](https://keras.io/api/layers/regularization_layers/dropout/), [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) and [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/).